In [0]:
df = spark.read\
    .option("header", True)\
    .option("inferSchema", True)\
    .csv("/Volumes/emp/default/sales_data/sales.csv")

display(df) 

from pyspark.sql.functions import col, to_date

# Remove duplicate rows
df = df.dropDuplicates()

# Convert data types
df = (
    df
    .withColumn("Order_Date", to_date("Order_Date"))
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("Price", col("Price").cast("double"))
)

print("Rows after cleaning:", df.count())

display(df)

Order_ID,Order_Date,Product,Category,City,Quantity,Price,Payment_Mode
1001,2026-09-01,Laptop,Electronics,Delhi,2,50000,UPI
1002,2026-09-01,Mouse,Accessories,Noida,5,500,Cash
1003,2026-09-02,Keyboard,Accessories,Delhi,3,1200,UPI
1004,2026-09-02,Laptop,Electronics,Noida,1,50000,Card
1005,2026-09-03,Monitor,Electronics,Ghaziabad,2,15000,UPI
1006,2026-09-03,Mouse,Accessories,Delhi,10,500,Cash
1007,2026-09-04,Keyboard,Accessories,Noida,4,1200,UPI
1008,2026-09-04,Laptop,Electronics,Ghaziabad,1,50000,Card
1009,2026-09-05,Monitor,Electronics,Delhi,3,15000,UPI
1010,2026-09-05,Mouse,Accessories,Ghaziabad,8,500,Cash


Rows after cleaning: 11


Order_ID,Order_Date,Product,Category,City,Quantity,Price,Payment_Mode
1008,2026-09-04,Laptop,Electronics,Ghaziabad,1,50000.0,Card
1002,2026-09-01,Mouse,Accessories,Noida,5,500.0,Cash
1006,2026-09-03,Mouse,Accessories,Delhi,10,500.0,Cash
1003,2026-09-02,Keyboard,Accessories,Delhi,3,1200.0,UPI
1007,2026-09-04,Keyboard,Accessories,Noida,4,1200.0,UPI
1011,2026-09-06,Laptop,Electronics,Delhi,2,50000.0,UPI
1001,2026-09-01,Laptop,Electronics,Delhi,2,50000.0,UPI
1009,2026-09-05,Monitor,Electronics,Delhi,3,15000.0,UPI
1010,2026-09-05,Mouse,Accessories,Ghaziabad,8,500.0,Cash
1004,2026-09-02,Laptop,Electronics,Noida,1,50000.0,Card


In [0]:
from pyspark.sql.functions import col
df = df.withColumn(
    "Revenue",
    col("Quantity") * col("Price")
)

display(df)

Order_ID,Order_Date,Product,Category,City,Quantity,Price,Payment_Mode,Revenue
1008,2026-09-04,Laptop,Electronics,Ghaziabad,1,50000.0,Card,50000.0
1002,2026-09-01,Mouse,Accessories,Noida,5,500.0,Cash,2500.0
1006,2026-09-03,Mouse,Accessories,Delhi,10,500.0,Cash,5000.0
1003,2026-09-02,Keyboard,Accessories,Delhi,3,1200.0,UPI,3600.0
1007,2026-09-04,Keyboard,Accessories,Noida,4,1200.0,UPI,4800.0
1011,2026-09-06,Laptop,Electronics,Delhi,2,50000.0,UPI,100000.0
1001,2026-09-01,Laptop,Electronics,Delhi,2,50000.0,UPI,100000.0
1009,2026-09-05,Monitor,Electronics,Delhi,3,15000.0,UPI,45000.0
1010,2026-09-05,Mouse,Accessories,Ghaziabad,8,500.0,Cash,4000.0
1004,2026-09-02,Laptop,Electronics,Noida,1,50000.0,Card,50000.0


In [0]:
from pyspark.sql.functions import sum as spark_sum, desc

product_revenue = (
    df.groupBy("Product")
      .agg(spark_sum("Revenue").alias("Total_Revenue"))
      .orderBy(desc("Total_Revenue"))
)

display(product_revenue)

Product,Total_Revenue
Laptop,300000.0
Monitor,75000.0
Mouse,11500.0
Keyboard,8400.0


In [0]:
city_revenue = (
    df.groupBy("City")
      .agg(spark_sum("Revenue").alias("Total_Revenue"))
      .orderBy(desc("Total_Revenue"))
)

display(city_revenue)

City,Total_Revenue
Delhi,253600.0
Ghaziabad,84000.0
Noida,57300.0


Databricks visualization. Run in Databricks to view.

In [0]:
best_selling = (
    df.groupBy("Product")
      .agg(spark_sum("Quantity").alias("Total_Quantity"))
      .orderBy(desc("Total_Quantity"))
)

display(best_selling)

Product,Total_Quantity
Mouse,23
Keyboard,7
Laptop,6
Monitor,5


In [0]:
total_revenue = df.agg(
    spark_sum("Revenue").alias("Total_Revenue")
)

display(total_revenue)

Total_Revenue
394900.0


In [0]:
df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("ecommerce_sales")

In [0]:
display(spark.table("ecommerce_sales"))

Order_ID,Order_Date,Product,Category,City,Quantity,Price,Payment_Mode,Revenue
1008,2026-09-04,Laptop,Electronics,Ghaziabad,1,50000.0,Card,50000.0
1002,2026-09-01,Mouse,Accessories,Noida,5,500.0,Cash,2500.0
1006,2026-09-03,Mouse,Accessories,Delhi,10,500.0,Cash,5000.0
1003,2026-09-02,Keyboard,Accessories,Delhi,3,1200.0,UPI,3600.0
1007,2026-09-04,Keyboard,Accessories,Noida,4,1200.0,UPI,4800.0
1011,2026-09-06,Laptop,Electronics,Delhi,2,50000.0,UPI,100000.0
1001,2026-09-01,Laptop,Electronics,Delhi,2,50000.0,UPI,100000.0
1009,2026-09-05,Monitor,Electronics,Delhi,3,15000.0,UPI,45000.0
1010,2026-09-05,Mouse,Accessories,Ghaziabad,8,500.0,Cash,4000.0
1004,2026-09-02,Laptop,Electronics,Noida,1,50000.0,Card,50000.0


In [0]:
spark.sql("DESCRIBE TABLE ecommerce_sales").show()

+------------+---------+-------+
|    col_name|data_type|comment|
+------------+---------+-------+
|    Order_ID|      int|   NULL|
|  Order_Date|     date|   NULL|
|     Product|   string|   NULL|
|    Category|   string|   NULL|
|        City|   string|   NULL|
|    Quantity|      int|   NULL|
|       Price|   double|   NULL|
|Payment_Mode|   string|   NULL|
|     Revenue|   double|   NULL|
+------------+---------+-------+



In [0]:
display(
    product_revenue
)

Product,Total_Revenue
Laptop,300000.0
Monitor,75000.0
Mouse,11500.0
Keyboard,8400.0


Databricks visualization. Run in Databricks to view.

Product,Total_Revenue
Laptop,300000.0
Monitor,75000.0
Mouse,11500.0
Keyboard,8400.0
